# Swedish Data & AI Job Market Analyzer
## Step 1: Explore the JobTech JobSearch API
Run the cells from top to bottom. We request **up to 100** advertisements for `data engineer`, explore the response, and save the original advertisements locally.

This is a keyword-search sample, not a complete Swedish job-market dataset. There is no date or country filter in this first request: do not assume all results are in Sweden or from the last six months. The API may return fewer than 100 ads.


## 1. Import libraries
`requests` calls the API, `pandas` creates tables, and `pathlib` handles Windows-compatible paths. JSON and time helpers support inspection and saving.

In [1]:
import requests
import pandas as pd
from pathlib import Path
import json
from datetime import datetime, timezone
from IPython.display import display


## 2. Request advertisements
The official endpoint accepts `q`, `limit`, and `offset`. A timeout prevents an indefinitely waiting request. If this cell fails, fix the reported problem and rerun it before continuing.

[Official JobTech API documentation](https://gitlab.com/arbetsformedlingen/job-ads/jobsearch-apis/-/blob/main/docs/GettingStartedJobSearchEN.md)


In [2]:
API_URL = "https://jobsearch.api.jobtechdev.se/search"
params = {"q": "data engineer", "limit": 100, "offset": 0}
# Clear previous results so a failed rerun cannot reuse an old download.
payload = None
advertisements = None

try:
    response = requests.get(
        API_URL, params=params, headers={"Accept": "application/json"}, timeout=30
    )
    response.raise_for_status()
except requests.exceptions.Timeout as exc:
    raise RuntimeError("The JobTech request timed out. Try running this cell again.") from exc
except requests.exceptions.ConnectionError as exc:
    raise RuntimeError(
        "Could not connect to JobTech. Check your internet connection and proxy settings."
    ) from exc
except requests.exceptions.HTTPError as exc:
    raise RuntimeError(
        f"JobTech returned HTTP {response.status_code}. Check the API URL and try again."
    ) from exc
except requests.exceptions.RequestException as exc:
    raise RuntimeError(f"The JobTech request failed: {exc}") from exc

try:
    payload = response.json()
except ValueError as exc:
    raise RuntimeError("JobTech returned a response that was not valid JSON.") from exc

if not isinstance(payload, dict) or not isinstance(payload.get("hits"), list):
    raise RuntimeError("Unexpected API response: expected an object with a 'hits' list.")
if not all(isinstance(ad, dict) for ad in payload["hits"]):
    raise RuntimeError("Unexpected API response: advertisements must be JSON objects.")
advertisements = payload["hits"]
downloaded_at = datetime.now(timezone.utc)


## 3. Check the result counts
`total.value` is the API-reported match count; `hits` contains this downloaded page. The total may be larger than the number downloaded. We also show the relation if the API provides one.

In [3]:
total = payload.get("total")
if isinstance(total, dict):
    total_matches = total.get("value", "Not provided")
    total_relation = total.get("relation")
else:
    total_matches = total if total is not None else "Not provided"
    total_relation = None

print(f"Total matching advertisements reported by API: {total_matches}")
if total_relation:
    print(f"Total count relation reported by API: {total_relation}")
print(f"Advertisements downloaded: {len(advertisements)}")
if len(advertisements) < params["limit"]:
    print("The API returned fewer than the requested 100 advertisements.")


Total matching advertisements reported by API: 76
Advertisements downloaded: 76
The API returned fewer than the requested 100 advertisements.


## 4. Create a DataFrame and preview 10 advertisements
Each row represents one advertisement. Nested objects remain dictionaries or lists so we can explore their structure.

In [4]:
df = pd.DataFrame(advertisements)
print(f"DataFrame shape: {df.shape}")
display(df.head(10))


DataFrame shape: (76, 39)


,relevance,id,external_id,original_id,label,webpage_url,logo_url,headline,application_deadline,number_of_vacancies,...,workplace_addresses,must_have,nice_to_have,application_contacts,publication_date,last_publication_date,removed,removed_date,source_type,timestamp
0,1.000000,31614367,None,None,[],https://arbetsformedlingen.se/platsbanken/anno...,https://arbetsformedlingen.se/rest/employer-lo...,Data engineer,2026-11-16T23:59:59,1,...,"[{'municipality': 'Solna', 'municipality_code'...","{'skills': [], 'languages': [], 'work_experien...","{'skills': [], 'languages': [], 'work_experien...","[{'name': 'Lee Gleichmann, Saco', 'description...",2026-10-19T13:57:54,2026-11-16T23:59:59,False,None,VIA_JOBPOSTING,1792411074177
1,1.000000,31613428,None,None,[],https://arbetsformedlingen.se/platsbanken/anno...,https://arbetsformedlingen.se/rest/employer-lo...,Data Engineer,2026-12-03T23:59:59,1,...,"[{'municipality': 'Göteborg', 'municipality_co...","{'skills': [], 'languages': [], 'work_experien...","{'skills': [], 'languages': [], 'work_experien...","[{'name': 'Amanda Blom', 'description': None, ...",2026-10-19T11:53:55,2026-12-03T23:59:59,False,None,VIA_JOBPOSTING,1792408291863
2,1.000000,31518118,None,None,[],https://arbetsformedlingen.se/platsbanken/anno...,NaN,Data Engineer,2026-10-24T23:59:59,1,...,"[{'municipality': 'Göteborg', 'municipality_co...","{'skills': [], 'languages': [], 'work_experien...","{'skills': [], 'languages': [], 'work_experien...","[{'name': 'Neethi Gopinadh', 'description': No...",2026-09-24T19:36:27,2026-10-24T23:59:59,False,None,VIA_JOBPOSTING,1790271387075
3,1.000000,30979729,None,None,[],https://arbetsformedlingen.se/platsbanken/anno...,NaN,Data Engineer,2026-10-27T23:59:59,1,...,"[{'municipality': 'Jönköping', 'municipality_c...","{'skills': [], 'languages': [], 'work_experien...","{'skills': [], 'languages': [], 'work_experien...",[],2026-04-30T11:53:20,2026-10-27T23:59:59,False,None,VIA_JOBPOSTING,1777573624056
4,0.986361,31588670,None,None,[],https://arbetsformedlingen.se/platsbanken/anno...,https://arbetsformedlingen.se/rest/employer-lo...,Data Engineer,2027-04-11T23:59:59,1,...,"[{'municipality': 'Stockholm', 'municipality_c...","{'skills': [], 'languages': [], 'work_experien...","{'skills': [], 'languages': [], 'work_experien...",[],2026-10-13T10:33:16,2027-04-11T23:59:59,False,None,VIA_JOBPOSTING,1791880396396
5,0.986361,31578878,None,None,[],https://arbetsformedlingen.se/platsbanken/anno...,NaN,Data Engineer,2026-10-22T23:59:59,1,...,"[{'municipality': 'Älmhult', 'municipality_cod...","{'skills': [], 'languages': [], 'work_experien...","{'skills': [], 'languages': [], 'work_experien...",[],2026-10-09T15:36:35,2026-10-22T23:59:59,False,None,VIA_ANNONSERA,1791552995683
6,0.986361,31574969,None,None,[],https://arbetsformedlingen.se/platsbanken/anno...,https://arbetsformedlingen.se/rest/employer-lo...,Data Engineer,2026-10-25T23:59:59,3,...,"[{'municipality': 'Malmö', 'municipality_code'...","{'skills': [], 'languages': [], 'work_experien...","{'skills': [], 'languages': [], 'work_experien...",[],2026-10-09T08:33:00,2026-10-25T23:59:59,False,None,VIA_ANNONSERA,1791527580676
7,0.986361,31554045,None,None,[],https://arbetsformedlingen.se/platsbanken/anno...,https://arbetsformedlingen.se/rest/employer-lo...,Data Engineer,2026-11-24T23:59:59,1,...,"[{'municipality': 'Stockholm', 'municipality_c...","{'skills': [], 'languages': [], 'work_experien...","{'skills': [], 'languages': [{'weight': 5, 'co...",[],2026-10-05T14:01:00,2026-11-24T23:59:59,False,None,VIA_ANNONSERA,1791983235720
8,0.986361,31539076,None,None,[],https://arbetsformedlingen.se/platsbanken/anno...,NaN,Data Engineer,2026-10-31T23:59:59,2,...,"[{'municipality': 'Stockholm', 'municipality_c...","{'skills': [], 'languages': [{'weight': 10, 'c...","{'skills': [], 'languages': [], 'work_experien...",[],2026-10-01T00:22:05,2026-10-31T23:59:59,False,None,VIA_ANNONSERA,1790806925698
9,0.986361,31207318,None,None,[],https://arbetsformedlingen.se/plats

## 5. Inspect all columns and data types
Pandas `object` columns can contain strings, dictionaries, lists, or missing values. Publication dates are still the original API strings here.

In [5]:
column_summary = df.dtypes.rename("dtype").rename_axis("column").reset_index()
with pd.option_context("display.max_rows", None):
    display(column_summary)


,column,dtype
0,relevance,float64
1,id,str
2,external_id,object
3,original_id,object
4,label,object
5,webpage_url,str
6,logo_url,str
7,headline,str
8,application_deadline,str
9,number_of_vacancies,int64


## 6. Build a simplified table
Use the documented fields `headline`, `employer.name`, `publication_date`, and `workplace_address.municipality` (falling back to city or region). Missing or null optional fields appear as empty values. Location is reported by the advertiser and needs validation in a future step.

In [6]:
def optional_object(value):
    return value if isinstance(value, dict) else {}

rows = []
for ad in advertisements:
    employer = optional_object(ad.get("employer"))
    address = optional_object(ad.get("workplace_address"))
    rows.append({
        "job_title": ad.get("headline"),
        "employer": employer.get("name"),
        "publication_date": ad.get("publication_date"),
        "location": address.get("municipality") or address.get("city") or address.get("region"),
    })

simple_df = pd.DataFrame(
    rows, columns=["job_title", "employer", "publication_date", "location"]
)
display(simple_df)


,job_title,employer,publication_date,location
0,Data engineer,Statens Skolverk,2026-10-19T13:57:54,Solna
1,Data Engineer,Eccera Professionals AB,2026-10-19T11:53:55,Göteborg
2,Data Engineer,Agile Resources AB,2026-09-24T19:36:27,Göteborg
3,Data Engineer,Techrytera AB,2026-04-30T11:53:20,Jönköping
4,Data Engineer,Newsec Property Asset Management Sweden AB,2026-10-13T10:33:16,Stockholm
...,...,...,...,...
71,Senior Data & AI Engineer till Spendrups,Academic Work Sweden AB,2026-10-16T08:21:20,Stockholm
72,Vi söker en Data Analyst/Analytics Engineer ti...,Academic Work Sweden AB,2026-10-20T13:17:17,Malmö
73,Performance Management / Data Mediation Engine...,Veritaz AB,2026-10-16T13:05:25,Stockholm
74,ML Engineer Machine Learning & Geospatial Dat...,Veritaz AB,2026-10-15T10:29:25,Malmö


## 7. Inspect one complete advertisement
Pretty-print the first advertisement to explore nested objects such as employer, workplace address, and description. Nothing is removed from this view.

In [7]:
if advertisements:
    print(json.dumps(advertisements[0], ensure_ascii=False, indent=2))
else:
    print("No advertisements were returned, so there is no example to inspect.")


{
  "relevance": 1.0,
  "id": "31614367",
  "external_id": null,
  "original_id": null,
  "label": [],
  "webpage_url": "https://arbetsformedlingen.se/platsbanken/annonser/31614367",
  "logo_url": "https://arbetsformedlingen.se/rest/employer-logo-api/api/v1/organisation/2021004185/logotyper/logo.png",
  "headline": "Data engineer",
  "application_deadline": "2026-11-16T23:59:59",
  "number_of_vacancies": 1,
  "description": {
    "text": "Vill du använda din kompetens och ditt engagemang till att skapa digitala lösningar för Skolsektorn i Sverige, samtidigt som du bidrar till Skolverkets digitala utveckling? Vi söker dig som är en erfaren och driven data engineer som kan arbeta både strategiskt och operativt med fokus på datalager och integration. Var med och bidra till skolans framtid!\nOm jobbet\nHos oss är arbete med data och analys centralt. Vi vill utöka vår förmåga inom dataintag och dataanalys och där har du en viktig roll. Som data engineer bidrar du både operativt och taktiskt

## 8. Save the original advertisements
Save the unmodified `hits` list as UTF-8 JSON, preserving Swedish characters and nested fields. Paths work when the notebook starts from either the project root or its `notebooks` folder. A UTC timestamp gives each download its own filename. Raw files are ignored by Git.

In [8]:
current_dir = Path.cwd().resolve()
if (current_dir / "requirements.txt").is_file() and (current_dir / "notebooks").is_dir():
    project_root = current_dir
elif current_dir.name == "notebooks" and (current_dir.parent / "requirements.txt").is_file():
    project_root = current_dir.parent
else:
    raise RuntimeError("Open Jupyter from the swedish-data-jobs folder or its notebooks folder.")

raw_dir = project_root / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
timestamp = downloaded_at.strftime("%Y%m%dT%H%M%S%fZ")
output_path = raw_dir / f"jobtech_data_engineer_{timestamp}.json"
with output_path.open("w", encoding="utf-8") as file:
    json.dump(advertisements, file, ensure_ascii=False, indent=2)

print(f"Success: saved {len(advertisements)} advertisements to:\n{output_path.resolve()}")


Success: saved 76 advertisements to:
C:\Users\zahir\Desktop\projects\swedish-data-jobs\data\raw\jobtech_data_engineer_20261021T122844375906Z.json
